# 14 - A completely novel attack: character-level typos (DeepWordBug)

TextFooler, PWWS and BERT-Attack all swap whole words for similar words. **DeepWordBug** (Gao et al., 2018) changes characters inside words: it swaps neighbouring letters, substitutes, deletes or inserts a letter, so words become misspellings. That is a different perturbation space. The TMD paper does not use it, none of our defenses was trained on it, and no DeepWordBug example is used anywhere in this project. It is built exactly as the authors' benchmark code builds it (their `build_english_attacker`), with the same limits as the other attacks: at most 30% of the words changed, 50 candidates per position, sentence similarity 0.84, query budget 50 x words.

Every defense we have is attacked once on the same 1000 test sentences, and the "attacks seen in training" column shows what each was trained on:

| Defense | Attacks seen | Notebook |
|---|---|---|
| vanilla BERT | none | 00 |
| TMD (our reproduction) | none | 00 |
| trained on TextFooler only | 1 | 10 |
| trained on BERT-Attack only | 1 | 11 |
| trained on TextFooler + BERT-Attack | 2 | 13 |
| trained on all three | 3 | 07b |

This answers two questions at once: who does better on an attack nobody has seen (TMD or ours), and whether seeing more attacks helps on a new one. A defense whose notebook has not been run is skipped.

**Cost (L4, rough):** about 3 to 4 hours (TMD is the slowest, since every query runs its random search). Set `N_TEST = 500` to halve it. Cached on Drive; resumes after a disconnect.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-13
N_TEST = 1000
ATTACK = 'deepwordbug'

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')

def guarded(prefix):
    for t in (f'{prefix}_s1', f'{prefix}_s2', f'{prefix}_s3', prefix):
        p = T.results_path(lab, t, N_TEST)
        if os.path.exists(p) and json.load(open(p))['clean']['acc'] > 90:
            return t

def load_enc(name, gan):
    p = lab._path(f'projenc_{name}_L{gan.latent_dim}C{gan.code_dim}.pt')
    if not os.path.exists(p):
        return None
    e = T.ProjEncoder(gan.latent_dim, gan.code_dim).to(T.DEV)
    e.load_state_dict(torch.load(p))
    return e.eval()

def victim(model, enc_name=None):
    # Rebuild a defense from its files on Drive: its BERT, GAN and encoder (no encoder: TMD's random search)
    lab.use_model(model)
    E_te, _ = lab.embeddings('test', model)
    if enc_name is None:
        if model == 'base':
            return T.Victim(lab, mode='none'), E_te
        return None, None
    E_tr_m, _ = lab.embeddings('train', model)
    E_va_m, _ = lab.embeddings('val', model)
    gan = lab.get_gan(model, E_tr_m, E_va_m)
    enc = load_enc(enc_name, gan)
    if enc is None:
        return None, None
    return T.Victim(lab, gan=gan, enc=enc, mode='tmd', proj_mode='encoder'), E_te

tf_tag, ba_tag = guarded('tf_only_enc1'), guarded('ba_only_enc1')
mix_tag = guarded('tf_ba_enc1')
suffix = lambda tag, prefix: tag[len(prefix):] if tag else ''
SPECS = [('vanilla', 'vanilla BERT', 0, 'base', None),
         ('baseline', 'TMD (our reproduction)', 0, 'base', 'TMD'),
         (tf_tag, 'trained on TextFooler only', 1, 'tfbert2', f"tf_enc1{suffix(tf_tag, 'tf_only_enc1')}" if tf_tag else None),
         (ba_tag, 'trained on BERT-Attack only', 1, 'babert2', f"ba_enc1{suffix(ba_tag, 'ba_only_enc1')}" if ba_tag else None),
         (mix_tag, 'trained on TextFooler + BERT-Attack', 2, 'mixbert2', f"mix_enc1{suffix(mix_tag, 'tf_ba_enc1')}" if mix_tag else None),
         ('advbert2_enc1', 'trained on all three attacks', 3, 'advbert2', 'b_enc1')]

## Attack every defense with DeepWordBug (one run each)

In [ ]:
rows = []
for tag, label, seen, model, enc_name in SPECS:
    if tag is None:
        print('skipping', label, '(its notebook has not been run)')
        continue
    if enc_name == 'TMD':
        lab.use_model('base')
        E_te, _ = lab.embeddings('test', 'base')
        gan = lab.get_gan('base', E_tr, E_va)
        v = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
    else:
        v, E_te = victim(model, enc_name)
        if v is None:
            print('skipping', label, '(files not found on Drive)')
            continue
    Y_te = lab.embeddings('test', model)[1]
    res = T.evaluate(lab, v, tag, E_te, Y_te, attacks=(ATTACK,), n=N_TEST, need_log=True)
    rows.append({'defense': label, 'attacks seen in training': seen, 'tag': tag,
                 'clean': round(res['clean']['acc'], 2), 'DeepWordBug': round(res[ATTACK]['acc'], 1)})
tab = pd.DataFrame(rows)
display(tab.drop(columns='tag'))

## Paired tests against TMD (same sentences) and the curve over attacks seen

In [ ]:
tests = [dict(T.mcnemar(lab, 'baseline', t, ATTACK, N_TEST), defense=l)
         for t, l in zip(tab.tag, tab.defense) if t not in ('baseline', 'vanilla')]
display(pd.DataFrame(tests)[['defense', 'examples', 'robust_only_a', 'robust_only_b', 'p']]
        .rename(columns={'robust_only_a': 'only TMD survives', 'robust_only_b': 'only this survives'}))
print(tab.groupby('attacks seen in training')['DeepWordBug'].max().to_string())
print('rows read: DeepWordBug accuracy under attack; the paper reports no number for this attack.')